# Quantum Error Correction: 3-Qubit Phase-Flip Code

## Overview

This notebook implements the **3-qubit phase-flip code** — the natural companion to the 3-qubit bit-flip code,
and the second building block of Shor's 9-qubit code.

Where the bit-flip code protects against $X$ errors in the $\{|0\rangle, |1\rangle\}$ basis,
the phase-flip code protects against $Z$ errors in the $\{|+\rangle, |-\rangle\}$ basis.
Understanding it deeply is essential before tackling Shor's code or CSS codes.

### What we cover

| Section | Content |
|---------|--------|
| **Theory 1** | What a phase-flip is and why it is invisible to the bit-flip code |
| **Theory 2** | The Hadamard trick — turning phase-flips into bit-flips |
| **Theory 3** | Encoding: the codewords and the circuit |
| **Theory 4** | Syndrome measurement in the Hadamard basis |
| **Theory 5** | Stabilizer formalism for the phase-flip code |
| **Theory 6** | Comparison: bit-flip vs phase-flip vs Shor |
| **Implementation** | Qiskit circuit, error injection, syndrome measurement |
| **Experiments** | Z errors on each qubit, X errors (undetectable), comparison plots |
| **Hardware** | IBM Quantum submission stub |

---
## Theory Part 1: What Is a Phase-Flip and Why Is It Dangerous?

### The Pauli-Z Gate

A phase-flip error is the action of the **Pauli-Z gate**:

$$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

Its effect on the computational basis states:
$$Z|0\rangle = |0\rangle, \qquad Z|1\rangle = -|1\rangle$$

For a general superposition $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$:
$$Z|\psi\rangle = \alpha|0\rangle - \beta|1\rangle$$

The **amplitudes are unchanged** — only the relative phase between $|0\rangle$ and $|1\rangle$ is flipped.

### Why the Bit-Flip Code is Blind to Phase-Flips

Recall the 3-qubit bit-flip codewords:
$$|0\rangle_L = |000\rangle, \qquad |1\rangle_L = |111\rangle$$

The bit-flip syndrome measures $Z_0Z_1$ and $Z_1Z_2$ (parity checks). Applying a Z error on qubit 0:
$$Z_0|000\rangle = |000\rangle \quad \text{(no change in bit values!)}$$
$$Z_0|111\rangle = -|111\rangle \quad \text{(only a global sign)}$$

The encoded state becomes $\alpha|000\rangle - \beta|111\rangle$ — the parity of every pair is still `00`.
**The syndrome reads zero. The error is completely invisible.**

### Phase-Flips in Physical Quantum Systems

Phase errors are extremely common in real hardware:

| Source | Mechanism |
|--------|----------|
| **Dephasing (T₂ noise)** | Fluctuating magnetic fields or charge noise shift the qubit frequency, randomising the phase |
| **Crosstalk** | Nearby qubits being driven shift the phase of idle qubits |
| **Gate miscalibration** | Small over/under-rotations in Z-axis gates accumulate phase errors |
| **Measurement backaction** | Partial measurement collapses phase information |

In many superconducting qubit architectures, **dephasing (T₂) is the dominant error channel**, making the phase-flip code highly relevant to real hardware.

---
## Theory Part 2: The Hadamard Trick — Rotating the Error Basis

### Two Bases for a Qubit

Every qubit lives in a 2-dimensional Hilbert space, but we can choose different orthonormal bases:

| Basis | States | Also called |
|-------|--------|------------|
| **Z-basis** | $|0\rangle, |1\rangle$ | Computational basis |
| **X-basis** | $|+\rangle = \frac{|0\rangle+|1\rangle}{\sqrt{2}},\ |-\rangle = \frac{|0\rangle-|1\rangle}{\sqrt{2}}$ | Hadamard basis |

### How Z and X Errors Transform Between Bases

The Hadamard gate $H$ **swaps** the X and Z bases:
$$H|0\rangle = |+\rangle, \quad H|1\rangle = |-\rangle$$
$$H|+\rangle = |0\rangle, \quad H|-\rangle = |1\rangle$$

Crucially, it also **conjugates** Pauli operators:
$$HZH = X, \qquad HXH = Z$$

This means:
- A **Z error** in the Z-basis looks like an **X error** (bit-flip) in the X-basis
- A **X error** in the Z-basis looks like a **Z error** (phase-flip) in the X-basis

### The Core Insight

We already know how to detect and correct X (bit-flip) errors using the 3-qubit repetition code.

If we apply Hadamards **before** encoding and **after** syndrome measurement, we **rotate our frame of reference** so that Z errors become X errors — and then we can use the exact same majority-vote machinery.

Concretely:

$$\underbrace{H^{\otimes 3}}_{\text{rotate to X-basis}} \circ \underbrace{\text{CNOT encoding}}_{\text{bit-flip code}} \circ \underbrace{H^{\otimes 3}}_{\text{rotate back to Z-basis}}$$

This is the entire conceptual difference between the bit-flip and phase-flip codes: **just a change of basis**.

---
## Theory Part 3: The Encoding

### Codewords

The phase-flip code encodes 1 logical qubit into 3 physical qubits using the **X-basis states** $|+\rangle$ and $|-\rangle$:

$$|0\rangle_L \;\rightarrow\; |{+}{+}{+}\rangle = \frac{1}{2\sqrt{2}}(|0\rangle+|1\rangle)^{\otimes 3}$$

$$|1\rangle_L \;\rightarrow\; |{-}{-}{-}\rangle = \frac{1}{2\sqrt{2}}(|0\rangle-|1\rangle)^{\otimes 3}$$

Expanding fully:
$$|0\rangle_L = \frac{1}{2\sqrt{2}}\big(|000\rangle+|001\rangle+|010\rangle+|011\rangle+|100\rangle+|101\rangle+|110\rangle+|111\rangle\big)$$
$$|1\rangle_L = \frac{1}{2\sqrt{2}}\big(|000\rangle-|001\rangle-|010\rangle+|011\rangle-|100\rangle+|101\rangle+|110\rangle-|111\rangle\big)$$

The logical information is stored in the **parity of the signs** across the three qubits — not in the bit values themselves.

### Encoding Circuit

The circuit has three steps:

**Step 1** — Copy the amplitude to all three qubits (identical to the bit-flip code start):
```
q0: ─■────■──    (CNOT control)
q1: ─X────┼──    (CNOT target)
q2: ──────X──    (CNOT target)
```
After this, we have $\alpha|000\rangle + \beta|111\rangle$ — the bit-flip codeword.

**Step 2** — Apply Hadamard to all three qubits to rotate into the X-basis:
```
q0: ─[H]─
q1: ─[H]─
q2: ─[H]─
```
The transformation $H^{\otimes 3}$ maps:
$$\alpha|000\rangle + \beta|111\rangle \;\rightarrow\; \alpha|{+}{+}{+}\rangle + \beta|{-}{-}{-}\rangle$$

This is the phase-flip codeword — a superposition of two states that differ only in their **relative phase pattern**.

### Why the Encoding Protects Against Z Errors

In the $|+\rangle/|-\rangle$ basis, a Z error acts as an X error:
$$Z|+\rangle = |-\rangle, \qquad Z|-\rangle = |+\rangle$$

So a Z error on qubit $i$ **flips** that qubit from $|+\rangle$ to $|-\rangle$ or vice versa — exactly a bit-flip in the Hadamard basis. The three-qubit majority vote can now detect and correct it, for the same reason it works for X errors in the computational basis.

---
## Theory Part 4: Syndrome Measurement in the Hadamard Basis

### The Challenge

In the bit-flip code, syndrome measurement was straightforward: measure the parity of adjacent qubit *bit values* using CNOT gates.

In the phase-flip code, the error manifests as a **sign difference**, not a bit-value difference. We cannot detect it with Z-basis parity checks — those would collapse the superposition and destroy the encoded information.

### The Solution: X-Basis Parity Checks

Instead of measuring $Z_i Z_j$ (bit parity), we measure $X_i X_j$ (phase parity).

The **X-parity** of two qubits asks: *do qubits $i$ and $j$ have the same phase?*
- $|{+}{+}\rangle$: both in $|+\rangle$, phases agree → X-parity = $+1$
- $|{-}{-}\rangle$: both in $|-\rangle$, phases agree → X-parity = $+1$
- $|{+}{-}\rangle$: phases disagree → X-parity = $-1$
- $|{-}{+}\rangle$: phases disagree → X-parity = $-1$

### Circuit for X-Parity Measurement

To measure $X_i X_j$ using an ancilla qubit $a$:

```
a:  ─[H]─●────●─[H]─[M]─
qi:       │    │
          X    │
qj:            X
```

1. Put ancilla in $|+\rangle$ with Hadamard
2. CNOT from ancilla onto $q_i$ and $q_j$ (ancilla is **control**, data qubits are **targets**)
3. Hadamard on ancilla, then measure

**Why this works**: in the X-basis, CNOT with the ancilla as control XORs the phase information into the ancilla without disturbing the logical content of the data qubits.

### Syndrome Table

Identical structure to the bit-flip code, but detecting Z errors instead of X errors:

| Syndrome $(s_1, s_0)$ | q0=q1 phase? | q1=q2 phase? | Error | Fix |
|----------------------|-------------|-------------|-------|-----|
| `00` | ✓ agree | ✓ agree | No error | — |
| `01` | ✗ differ | ✓ agree | Z on qubit 0 | Apply $Z_0$ |
| `11` | ✗ differ | ✗ differ | Z on qubit 1 | Apply $Z_1$ |
| `10` | ✓ agree | ✗ differ | Z on qubit 2 | Apply $Z_2$ |

### What About X Errors?

An X error on qubit $i$ flips $|0\rangle \leftrightarrow |1\rangle$. In the $|+\rangle/|-\rangle$ encoded state, this is invisible to $X_i X_j$ parity checks — X errors commute with X-parity operators.  
The phase-flip code is **completely blind to bit-flip (X) errors** — the exact opposite of the bit-flip code.  
This complementary blindness is precisely why Shor needed to concatenate both codes.

---
## Theory Part 5: Stabilizer Formalism

The phase-flip code is a **[[3, 1, 3]] stabilizer code** — same parameters as the bit-flip code, but different stabilizers.

### Stabilizer Generators

The code space is defined by two stabilizers:

$$S_1 = X_0 X_1 I, \qquad S_2 = I X_1 X_2$$

Contrast with the bit-flip code which uses $Z_0Z_1$ and $Z_1Z_2$. The roles of X and Z are perfectly swapped.

**Verification** — both codewords are $+1$ eigenstates of both stabilizers:

$$S_1|{+}{+}{+}\rangle = X_0X_1|{+}{+}{+}\rangle = |{+}{+}{+}\rangle \quad \checkmark$$
$$S_1|{-}{-}{-}\rangle = X_0X_1|{-}{-}{-}\rangle = |{-}{-}{-}\rangle \quad \checkmark$$

(Since $X|+\rangle = |+\rangle$ and $X|-\rangle = -|-\rangle$... wait — both states are stabilized because $X_0X_1$ flips both qubits: $(-1)(-1) = +1$ for $|{-}{-}\rangle$.)

### Logical Operators

$$\bar{Z} = Z_0 Z_1 Z_2 \quad \text{(logical phase-flip — flips } |0\rangle_L \leftrightarrow |1\rangle_L\text{)}$$
$$\bar{X} = X_0 \quad \text{(logical bit-flip — or any single } X_i\text{)}$$

### Error Detection via Anti-Commutation

A Z error $Z_i$ anti-commutes with any $X_j$ at the same site (since $ZX = -XZ$).  
This changes the eigenvalue of the stabilizers involving qubit $i$ from $+1$ to $-1$:

| Error | Anti-commutes with | Syndrome |
|-------|-------------------|----------|
| $Z_0$ | $S_1 = X_0X_1$ | $S_1 = -1,\ S_2 = +1$ → `01` |
| $Z_1$ | $S_1, S_2$ | $S_1 = -1,\ S_2 = -1$ → `11` |
| $Z_2$ | $S_2 = X_1X_2$ | $S_1 = +1,\ S_2 = -1$ → `10` |

### The X/Z Code Duality

| Property | Bit-flip code | Phase-flip code |
|----------|--------------|----------------|
| Stabilizers | $Z_0Z_1,\ Z_1Z_2$ | $X_0X_1,\ X_1X_2$ |
| Logical $\bar{X}$ | $X_0X_1X_2$ | $X_0$ |
| Logical $\bar{Z}$ | $Z_0$ | $Z_0Z_1Z_2$ |
| Corrects | X (bit-flip) errors | Z (phase-flip) errors |
| Blind to | Z (phase-flip) errors | X (bit-flip) errors |
| Codewords in | Z-basis: $|000\rangle, |111\rangle$ | X-basis: $|{+}{+}{+}\rangle, |{-}{-}{-}\rangle$ |

This perfect duality makes the phase-flip code the **Hadamard-conjugate** of the bit-flip code.

---
## Theory Part 6: Where the Phase-Flip Code Fits in QEC

### The Three-Code Progression

| Code | Protects | Blind to | Key idea |
|------|----------|----------|----------|
| 3-qubit bit-flip | X errors | Z errors | Majority vote in Z-basis |
| **3-qubit phase-flip** | **Z errors** | **X errors** | **Majority vote in X-basis (via Hadamard)** |
| Shor 9-qubit | X and Z errors | — | Concatenate both |

### The Path from Here to Shor's Code

The phase-flip code is the **outer code** in Shor's construction:

1. Apply the phase-flip code: encode into three $|+\rangle/|-\rangle$ blocks
2. Apply the bit-flip code to each block: protect each $|\pm\rangle$ against X errors
3. Result: $3 \times 3 = 9$ qubits that correct both error types

Understanding the phase-flip code is therefore the conceptual bridge between the two 3-qubit codes and Shor's complete solution.

---

> **Key Takeaway**: The phase-flip code is structurally identical to the bit-flip code, but operates in the conjugate (Hadamard) basis. A Z error in the computational basis becomes an X error in the Hadamard basis, where the same majority-vote machinery applies. The two codes are exact Hadamard-conjugates of each other, and their concatenation is the essence of Shor's 9-qubit code.

In [ ]:
# Install required packages
!pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.visualization import plot_histogram
from IPython.display import display

print("All imports successful.")

---
## Implementation

### Circuit Layout

```
Data qubits:    q[0]  q[1]  q[2]
Ancilla qubits: a[0]  a[1]
Classical bits: s[0]  s[1]   (syndrome)
```

The circuit has four stages:
1. **Prepare** the logical qubit on q[0]
2. **Encode** with CNOT gates (copy) + Hadamard gates (rotate to X-basis)
3. **Inject** an optional Z or X error
4. **Measure syndrome** with X-parity checks on the ancilla qubits

In [ ]:
def build_phase_flip_circuit(z_error=None, x_error=None, initial_state='0'):
    """
    Build the 3-qubit phase-flip error detection circuit.

    Parameters:
        z_error      (int|None): qubit index (0, 1, or 2) to apply Z error, or None
        x_error      (int|None): qubit index (0, 1, or 2) to apply X error, or None
                                  (undetectable — demonstrates code blindness)
        initial_state (str):     '0' or '1' — logical qubit to encode

    Returns:
        QuantumCircuit: full circuit with 3 data + 2 ancilla qubits
    """
    data    = QuantumRegister(3, 'q')
    ancilla = QuantumRegister(2, 'a')
    creg    = ClassicalRegister(2, 's')
    qc      = QuantumCircuit(data, ancilla, creg)

    # ── Step 1: Prepare logical qubit ────────────────────────────
    if initial_state == '1':
        qc.x(data[0])  # Start in |1>

    # ── Step 2: Encode ───────────────────────────────────────────
    # First: spread amplitude via CNOT (same as bit-flip code)
    # alpha|000> + beta|100>  →  alpha|000> + beta|110>  →  alpha|000> + beta|111>
    qc.cx(data[0], data[1])
    qc.cx(data[0], data[2])

    # Second: Hadamard on all three qubits to rotate into X-basis
    # |000> → |+++>,  |111> → |--->
    # So: alpha|000> + beta|111>  →  alpha|+++> + beta|--->
    qc.h(data[0])
    qc.h(data[1])
    qc.h(data[2])

    qc.barrier(label='encoded')

    # ── Step 3: Inject errors ────────────────────────────────────
    if z_error is not None:
        qc.z(data[z_error])  # Phase-flip on specified qubit
    if x_error is not None:
        qc.x(data[x_error])  # Bit-flip on specified qubit (undetectable)

    if z_error is not None or x_error is not None:
        qc.barrier(label='error')

    # ── Step 4: Syndrome measurement (X-parity checks) ───────────
    #
    # We measure X_0 X_1 (do q0 and q1 have the same phase?)
    # and       X_1 X_2 (do q1 and q2 have the same phase?)
    #
    # Circuit for each X-parity check:
    #   H on ancilla  →  CNOT(ancilla→q_i)  →  CNOT(ancilla→q_j)  →  H on ancilla  →  measure
    #
    # The ancilla starts in |0>, Hadamard puts it in |+>,
    # CNOTs accumulate the XOR of the X-eigenvalues of q_i and q_j,
    # final Hadamard maps back to Z-basis for readout.

    # Syndrome bit 0: X-parity of q[0] and q[1]
    qc.h(ancilla[0])
    qc.cx(ancilla[0], data[0])
    qc.cx(ancilla[0], data[1])
    qc.h(ancilla[0])

    # Syndrome bit 1: X-parity of q[1] and q[2]
    qc.h(ancilla[1])
    qc.cx(ancilla[1], data[1])
    qc.cx(ancilla[1], data[2])
    qc.h(ancilla[1])

    qc.barrier(label='syndrome')

    # Measure ancilla
    qc.measure(ancilla[0], creg[0])
    qc.measure(ancilla[1], creg[1])

    return qc


# Visualise the circuit with a Z error on qubit 1
qc_demo = build_phase_flip_circuit(z_error=1)
print("3-Qubit Phase-Flip Code Circuit (Z error on q1):")
display(qc_demo.draw('mpl'))

In [ ]:
def decode_syndrome(syndrome_str):
    """
    Decode a 2-bit syndrome string into a Z correction prescription.

    Qiskit returns syndrome as s[1]s[0] (MSB on left).
    We reverse to index from 0.

    Syndrome table:
        00 → no error
        01 → Z on qubit 0  (s[0]=1, s[1]=0: q0≠q1, q1=q2)
        11 → Z on qubit 1  (s[0]=1, s[1]=1: q0≠q1, q1≠q2)
        10 → Z on qubit 2  (s[0]=0, s[1]=1: q0=q1, q1≠q2)

    Returns:
        int|None: qubit index to apply correction Z, or None
    """
    s = syndrome_str[::-1]  # s[0] is now index 0
    table = {
        '00': None,
        '01': 0,
        '11': 1,
        '10': 2,
    }
    return table.get(s)


# Verify the decoder
test_cases = [
    ('00', 'No error'),
    ('01', 'Z on q0'),
    ('11', 'Z on q1'),
    ('10', 'Z on q2'),
]
print(f"{'Syndrome':<12} {'Expected':<15} {'Decoded correction'}")
print("-" * 45)
for syn, label in test_cases:
    correction = decode_syndrome(syn)
    print(f"{syn:<12} {label:<15} Z → q{correction}")

---
## Experiments
### Experiment 1: Phase-Flip (Z) Error Detection on Every Qubit

In [ ]:
simulator = AerSimulator()

def run_experiment(z_error=None, x_error=None, initial_state='0', shots=1024):
    """Run the phase-flip code and return counts, top syndrome, and correction."""
    qc = build_phase_flip_circuit(z_error=z_error, x_error=x_error,
                                   initial_state=initial_state)
    result  = simulator.run(qc, shots=shots).result()
    counts  = result.get_counts()
    top_syn = max(counts, key=counts.get)
    correction = decode_syndrome(top_syn)
    return counts, top_syn, correction


print("Phase-Flip (Z) Error Detection")
print("=" * 48)
print(f"{'Error':<12} {'Syndrome':<12} {'Correction'}")
print("-" * 48)

# No error baseline
counts, syn, corr = run_experiment()
print(f"{'None':<12} {syn:<12} {f'Z→q{corr}' if corr is not None else 'None (correct)'}")

# Z error on each qubit
for q in range(3):
    counts, syn, corr = run_experiment(z_error=q)
    print(f"{'Z on q'+str(q):<12} {syn:<12} {f'Z→q{corr}' if corr is not None else '—'}")

### Experiment 2: Bit-Flip (X) Errors — Demonstrating Code Blindness

An X error should produce syndrome `00` (no error detected) on every qubit —
confirming the phase-flip code is completely blind to bit-flip errors.

In [ ]:
print("Bit-Flip (X) Error — Expected: syndrome '00' every time (undetectable)")
print("=" * 60)
print(f"{'Error':<12} {'Syndrome':<12} {'Detected?'}")
print("-" * 60)

for q in range(3):
    counts, syn, corr = run_experiment(x_error=q)
    detected = "⚠ YES (unexpected)" if syn != '00' else "✗ No (expected)"
    print(f"{'X on q'+str(q):<12} {syn:<12} {detected}")

print()
print("Result: all X errors produce syndrome 00 — the phase-flip code cannot see them.")
print("This is why Shor's code is needed: concatenate with the bit-flip code to cover both.")

### Experiment 3: Syndrome Histogram — No Error vs Z Error vs X Error

In [ ]:
counts_none, _, _ = run_experiment()
counts_z1,   _, _ = run_experiment(z_error=1)
counts_x1,   _, _ = run_experiment(x_error=1)

fig = plot_histogram(
    [counts_none, counts_z1, counts_x1],
    legend=['No error', 'Z on q1', 'X on q1 (undetectable)'],
    title='Phase-Flip Code: Syndrome Comparison',
    figsize=(9, 4)
)
display(fig)
print("'11' → Z error on qubit 1 detected | '00' → no error (or undetectable X error)")

### Experiment 4: Syndrome Map for All Single-Qubit Z Errors

In [ ]:
syndrome_map = {}
labels       = ['No error', 'Z on q0', 'Z on q1', 'Z on q2']
scenarios    = [(None, None), (0, None), (1, None), (2, None)]

for label, (z_err, _) in zip(labels, scenarios):
    _, syn, _ = run_experiment(z_error=z_err)
    syndrome_map[label] = syn

fig, ax = plt.subplots(figsize=(8, 3))
syndrome_vals = [int(s, 2) for s in syndrome_map.values()]
bars = ax.bar(labels, syndrome_vals, color=['steelblue', 'tomato', 'tomato', 'tomato'],
              edgecolor='black', linewidth=0.8)

for bar, syn in zip(bars, syndrome_map.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'`{syn}`', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Syndrome (decimal)', fontsize=11)
ax.set_title('Syndrome Value per Error Scenario', fontsize=13, fontweight='bold')
ax.set_ylim(0, 4)
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['0 (00)', '1 (01)', '2 (10)', '3 (11)'])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Bonus: Depolarising Noise Simulation

Sweep over physical error rates and compare:
- **Unprotected qubit**: probability of measuring `0` correctly from $|0\rangle$ under noise
- **Phase-flip code**: fraction of shots with a clean `00` syndrome under noise (no error detected)

In [ ]:
def unprotected_fidelity(error_rate, shots=2048):
    qc = QuantumCircuit(1, 1)
    qc.measure(0, 0)
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(error_rate, 1), ['id'])
    result = AerSimulator(noise_model=nm).run(qc, shots=shots).result()
    return result.get_counts().get('0', 0) / shots


def phase_flip_clean_fraction(error_rate, shots=2048):
    qc = build_phase_flip_circuit()  # no injected error
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(error_rate, 1),
                                   ['h', 'x', 'z', 'id'])
    nm.add_all_qubit_quantum_error(depolarizing_error(error_rate * 2, 2), ['cx'])
    result = AerSimulator(noise_model=nm).run(qc, shots=shots).result()
    return result.get_counts().get('00', 0) / shots


error_rates   = np.linspace(0.001, 0.06, 16)
raw_fids      = [unprotected_fidelity(p) for p in error_rates]
pfc_fractions = [phase_flip_clean_fraction(p) for p in error_rates]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(error_rates * 100, raw_fids, 'o-', color='crimson',
        label='Unprotected qubit fidelity')
ax.plot(error_rates * 100, pfc_fractions, 's-', color='steelblue',
        label='Phase-flip code: clean syndrome fraction')
ax.set_xlabel('Depolarising error rate per gate (%)', fontsize=12)
ax.set_ylabel('Fraction', fontsize=12)
ax.set_title('Unprotected vs Phase-Flip Code Under Noise', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The phase-flip code circuit is larger (more gates), so it accumulates more")
print("noise than a single qubit at the same physical error rate.")
print("This motivates fault-tolerant thresholds: QEC helps only when physical errors")
print("are below ~1% for most codes.")

---
## Running on IBM Quantum Hardware

**Prerequisites:**
1. Free account at [https://quantum.ibm.com](https://quantum.ibm.com)
2. Copy your API token from the IBM Quantum dashboard
3. Paste it below and set `RUN_ON_HARDWARE = True`

> **Note**: The phase-flip code uses 5 qubits (3 data + 2 ancilla) and is well within
> the limits of any IBM Quantum device. On real hardware you will observe non-zero syndromes
> even without injected errors — this is real dephasing (T₂) noise in action.

In [ ]:
RUN_ON_HARDWARE = False   # Set to True to submit to IBM Quantum
IBM_API_TOKEN   = "YOUR_TOKEN_HERE"
BACKEND_NAME    = "ibm_sherbrooke"  # or 'ibm_brisbane', 'ibm_kyoto', etc.

if RUN_ON_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    service = QiskitRuntimeService(channel='ibm_quantum', token=IBM_API_TOKEN)
    backend = service.backend(BACKEND_NAME)
    print(f"Connected to: {backend.name}  ({backend.num_qubits} qubits)")

    # Build circuit with Z error on qubit 1
    qc_hw = build_phase_flip_circuit(z_error=1)

    # Transpile
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
    qc_t = pm.run(qc_hw)
    print(f"Transpiled depth: {qc_t.depth()}  |  2Q gates: {qc_t.num_nonlocal_gates()}")

    # Submit
    job = Sampler(backend).run([qc_t], shots=1024)
    print(f"Job ID: {job.job_id()}")
    print("Check status: https://quantum.ibm.com/jobs")

    # Retrieve
    hw_counts = job.result()[0].data.s.get_counts()
    display(plot_histogram(hw_counts,
                           title="IBM Hardware — Phase-Flip Code (Z error on q1)"))
else:
    print("Hardware execution disabled.")
    print("Set RUN_ON_HARDWARE = True and add your IBM Quantum API token to run on real hardware.")

---
## Summary

### What We Built

| Component | Description |
|-----------|-------------|
| Encoding circuit | CNOT pair + H⊗3 rotating to $|{+}{+}{+}\rangle / |{-}{-}{-}\rangle$ codewords |
| Z error injection | Configurable phase-flip on any of the 3 data qubits |
| X error injection | Demonstrates code blindness to bit-flip errors |
| Syndrome measurement | X-parity checks via H–CNOT–CNOT–H on 2 ancilla qubits |
| Decoder | Maps 2-bit syndrome to Z correction prescription |
| Noise experiment | Depolarising noise sweep, raw vs phase-flip protected |
| Hardware runner | IBM Quantum submission stub |

### Key Results Observed

- Z errors on q0, q1, q2 produce unique syndromes `01`, `11`, `10` respectively ✅
- X errors (bit-flips) produce syndrome `00` — completely undetected ✅ (expected)
- Under depolarising noise, the larger circuit accumulates more noise than a single qubit

### Notebook Series Progress

| Notebook | Topic | Status |
|----------|-------|--------|
| 1 | 3-Qubit Bit-Flip Code | ✅ Complete |
| 2 | **3-Qubit Phase-Flip Code** | ✅ **This notebook** |
| 3 | Shor's 9-Qubit Code | 🔲 Next |
| 4 | Steane [[7,1,3]] Code | 🔲 Upcoming |
| 5 | Surface Code | 🔲 Upcoming |